# 🚗 Proyecto Final de Máster en IA: Aprendizaje por Refuerzo y Conducción Autónoma

Bienvenidos al reto final de Aprendizaje por Refuerzo (Deep RL). Vuestro objetivo es entrenar un agente capaz de conducir un vehículo autónomo en el simulador 3D **Duckietown**, utilizando únicamente datos de píxeles en bruto (cámara del salpicadero).

Un agente que memoriza un circuito es inútil. Deberéis entrenar en múltiples mapas y vuestra nota final dependerá del rendimiento del agente en un **mapa oculto** con obstáculos estáticos.

### 📋 Fases del Proyecto

* **Fase 1: Calentamiento.** Implementar Q-Learning tabular desde cero en `FrozenLake-v1`.
* **Fase 2: Baselines (DQN y PPO).** Implementar dos baselines usando la librería Stable Baselines3. Dado que Duckietown usa acciones continuas, tendréis que crear un Wrapper para discretizarlas en DQN.
* **Fase 3: Algoritmo Avanzado.** Diseñar y ajustar un tercer algoritmo de libre elección (ej. SAC, TD3, A2C, o una versión fuertemente optimizada de PPO con Curriculum Learning) que supere a los baselines.

### ⚠️ EL CONTRATO DE EVALUACIÓN (LEER ATENTAMENTE)
La evaluación está automatizada. Si vuestro código no funciona a la primera en la máquina del profesor, la nota de código será 0.
1.  **Observaciones:** El modelo DEBE esperar un espacio de observación con forma `(1, 64, 64)` apilado en 4 frames (Frame Stacking).
2.  **Clases personalizadas:** Vuestro archivo de evaluación debe incluir las definiciones exactas de las clases de Python de vuestra CNN y Wrappers.
3.  **Dry-Run:** Antes de entregar, debéis ejecutar todo en un Google Colab limpio para verificar que las dependencias funcionan.


---

> **Entrega autosuficiente (cómo ejecutar).** Este proyecto se entrega como ZIP que ya
> contiene todo (código, `requirements.txt`, `best_agent.zip` en la raíz, `Report.pdf`).
> En Colab: sube el ZIP y ejecuta las celdas en orden — la primera **descomprime el ZIP
> automáticamente** si hace falta, monta un **venv Python 3.11** e instala el stack. No
> hay que clonar GitHub ni subir el modelo aparte; `best_agent.zip` se copia solo a
> `models/`. El modelo final entregado es **PPO 20k**.

## Dependencias y entorno (Colab)

El kernel de Colab es **Python 3.12**, pero el stack (numpy 1.23.5, gym 0.25.2,
Duckietown…) requiere **Python 3.11**. Por eso se monta un **venv 3.11** y se ejecuta
todo con `{PY}` (subprocess), no con el `pip`/`python` del kernel. **Ejecutar en orden.**

In [ ]:
# Usar los archivos LOCALES del ZIP de entrega (NO se clona GitHub por defecto).
# Si solo está MAML_entrega_final.zip subido (sin descomprimir), se descomprime solo.
import os
import zipfile

def _has_project(d):
    return (os.path.isfile(os.path.join(d, "requirements.txt"))
            and os.path.isfile(os.path.join(d, "train.py"))
            and os.path.isfile(os.path.join(d, "eval.py"))
            and os.path.isdir(os.path.join(d, "src")))

def _find_project():
    return next((d for d in ["/content/MAML", "/content", os.getcwd()]
                 if _has_project(d)), None)

PROJECT = _find_project()

# Si no se encuentra el proyecto, intentar descomprimir el ZIP de entrega.
if PROJECT is None:
    zip_path = next((z for z in ["/content/MAML_entrega_final.zip",
                                 os.path.join(os.getcwd(), "MAML_entrega_final.zip")]
                     if os.path.isfile(z)), None)
    if zip_path is not None:
        # Solo creamos/usamos /content/MAML si no es ya un proyecto válido.
        if not _has_project("/content/MAML"):
            os.makedirs("/content/MAML", exist_ok=True)
            print(f"Descomprimiendo {zip_path} en /content/MAML ...")
            with zipfile.ZipFile(zip_path) as zf:
                zf.extractall("/content/MAML")
        PROJECT = _find_project()

if PROJECT is None:
    print("[ERROR] No se encontraron los archivos del proyecto "
          "(requirements.txt, train.py, eval.py, src/) ni MAML_entrega_final.zip.")
    print("Sube y/o descomprime el ZIP de entrega como /content/MAML y reejecuta.")
    print("Fallback de desarrollo (solo si NO tienes el ZIP): descomenta el git clone:")
    # !git clone https://github.com/AdolfoPM02/MAML.git /content/MAML
    raise FileNotFoundError("Proyecto no encontrado: descomprime el ZIP como /content/MAML.")

print("Proyecto encontrado en:", PROJECT)
%cd $PROJECT
!ls

In [ ]:
# a) Python 3.11 + dependencias de sistema (Duckietown headless)
!sudo add-apt-repository -y ppa:deadsnakes/ppa
!sudo apt-get update -qq
!sudo apt-get install -y -qq python3.11 python3.11-venv python3.11-dev xvfb python3-opengl ffmpeg freeglut3-dev

In [ ]:
# b) Crear el venv 3.11 y definir PY (intérprete del venv usado en TODO el notebook)
!python3.11 -m venv /content/venv-maml
PY = "/content/venv-maml/bin/python"
!{PY} -m pip install -U pip wheel setuptools
!{PY} --version

In [ ]:
# c) Instalar el stack EXACTO con el Python 3.11 del venv (NO el kernel 3.12)
%cd $PROJECT
!{PY} -m pip install -r requirements.txt
# gym-duckietown se instala APARTE y SIN deps (sus dependencias ya van en requirements.txt)
!{PY} -m pip install --no-deps "duckietown-gym-daffy @ git+https://github.com/duckietown/gym-duckietown.git@c76a7fec7f739db4f624f40ca83ce99383665558"
# Re-fijar numpy al final por si alguna dependencia lo cambió
!{PY} -m pip install --force-reinstall --no-deps numpy==1.23.5

In [ ]:
# d) Verificación de imports y versiones
!MPLBACKEND=Agg {PY} -c "import numpy, torch, gym, gymnasium, stable_baselines3; import gym_duckietown; print('numpy', numpy.__version__); print('torch', torch.__version__); print('gym', gym.__version__); print('gymnasium', gymnasium.__version__); print('stable_baselines3', stable_baselines3.__version__); print('import gym_duckietown OK')"

## Fase 1: Calentamiento Tabular (Q-Learning)
Demostrad que entendéis la Ecuación de Bellman. Implementad Q-Learning desde cero (sin librerías de Deep RL) para resolver el entorno `FrozenLake-v1` (cuadrícula 8x8, resbaladiza).


La implementación completa (sin librerías de Deep RL) está en **`q_learning_sandbox.py`**. La celda siguiente la ejecuta (imprime métricas y guarda la curva de media móvil). Celda **opcional** (~30 s).

In [ ]:
!MPLBACKEND=Agg {PY} q_learning_sandbox.py

## Fases 2 y 3: Conducción Autónoma en Duckietown

En esta sección, construiréis el pipeline para DQN, PPO y vuestro **Algoritmo Avanzado (Fase 3)**.
Aquí tenéis el código base para inicializar la pantalla virtual que requiere Google Colab y el esqueleto de procesamiento de imágenes.

**Mapas de Entrenamiento Permitidos:**
`Duckietown-loop_empty-v0`, `Duckietown-udem1-v0`, `Duckietown-zigzag_dists-v0`, `Duckietown-small_loop-v0`, `Duckietown-straight_road-v0`.

**Nuestra solución** implementa el pipeline en módulos importables (contrato: clases definidas y cargables):
* `src/wrappers.py` → `DuckieWrapper` (RGB→recorte→grises→64×64→`(1,64,64)`) y `DiscreteActionWrapper` (DQN).
* `src/cnn.py` → `CustomCNN`; `src/envs.py` → `build_vec_env` con **FrameStack(4)** → `(4,64,64)`.
* `train.py` entrena DQN/PPO/SAC; `eval.py` evalúa. El lanzador `scripts/run_training_plan.py`
orquesta los experimentos.

Celdas en **dry-run** (muestran los comandos, **no entrenan**) — PPO y DQN (Fase 2) y SAC (Fase 3):

In [ ]:
# Fase 2 (baselines): PPO y DQN
!{PY} scripts/run_training_plan.py --stage ppo20k --dry-run --eval-after
!{PY} scripts/run_training_plan.py --stage dqn20k --dry-run --eval-after
# Fase 3 (algoritmo avanzado): SAC
!{PY} scripts/run_training_plan.py --stage sac20k --dry-run --eval-after

### Bucle de Entrenamiento
Entrenad vuestros modelos (DQN, PPO y el Algoritmo Avanzado). Recordad usar **Frame Stacking** (apilamiento de frames) para que el agente tenga percepción de la velocidad y giros. Guardad vuestro modelo final usando `model.save()`.

**Nota:** los entrenamientos largos **no son obligatorios** para la corrección (el profesor solo carga `best_agent.zip` y evalúa). Están documentados en **`EXPERIMENTS.md`** y se lanzan con `scripts/run_training_plan.py` (`--init-order model-first`, CPU/GPU); el modelo se guarda con `model.save()`.

## Resultados finales
Entrenamiento real en Colab (GPU) con `--init-order model-first`. Evaluación con `eval.py`
(3 episodios por mapa). Celda = `recompensa media ± std / longitud media`.

| modelo | loop_empty | small_loop | loop_obstacles | decisión |
|---|---|---|---|---|
| DQN 20k | −1013.4 / 112 | −1001.9 / 86 | −1065.8 / 61 | descartado |
| DQN 50k | −1047.6 / 82 | −1016.6 / 74 | −1032.0 / 37 | descartado |
| **PPO 20k** | **961.0 / 1500** | **317.5 / 1500** | **1118.2 / 1500** | **GANADOR** |
| PPO 50k | −813.7 / 1500 | −1238.9 / 1500 | −1105.4 / 1500 | descartado |
| SAC 20k | 990.6 / 1500 | −1253.6 / 296 | 915.2 / 1500 | no gana |
| SAC 50k | — | — | — | no completado |
| PPO avanzado v1 (ppo_adv) | −997.1 | −1194.9 | −1538.9 | descartado |
| PPO avanzado v2 (ppo_adv_v2) | −188.3 | −84.7 | 138.7 | no gana |
| PPO fine-tuned (ppo_ft) | −364.2 | 821.3 | 513.2 | no gana |

**Modelo final: `best_agent.zip` = PPO 20k** (`ppo_loop_empty_20k_gpu`): mejor recompensa en
el mapa oculto `loop_obstacles` (1118.2) y positiva en los tres mapas.

**Fase 3 (honesto):** se implementaron y evaluaron varias variantes avanzadas —**PPO
avanzado v1** (hiperparámetros agresivos), **PPO avanzado v2** (conservador, `ent_coef=0.001`)
y **PPO fine-tuning** desde el ganador (`--init-model`, `lr=5e-5`)— además de **SAC**. **Ninguna
superó al PPO 20k** en el criterio principal (robustez/recompensa en `loop_obstacles`), por lo
que el modelo final se mantiene como PPO 20k. SAC 50k no se completó (interrumpido en Colab).

> **Nota:** `Duckietown-loop_obstacles-v0` se usó **solo para evaluación**
> (`--allow-eval-hidden`), **nunca para entrenamiento** (bloqueado por `ValueError`).

## Evaluación Visual y Generación de Video
Para que podáis ver a vuestro agente conducir, ejecutad esta celda. Creará un entorno de prueba, dejará que el agente conduzca y generará un vídeo MP4. **El profesor usará un código idéntico a este en el mapa oculto para puntuaros.**

**Nota:** la generación de **vídeo es opcional**; la **validación cuantitativa** se hace con `eval.py` (`--init-order model-first`), que es más robusta y coincide con nuestro contrato funcional. La siguiente celda **copia automáticamente** `best_agent.zip` (= **PPO 20k**) desde la raíz a `models/best_agent.zip` y evalúa.

In [ ]:
# Preparar best_agent.zip automáticamente (viene en la raíz del ZIP de entrega)
%cd $PROJECT

import os
import shutil

os.makedirs("models", exist_ok=True)

if os.path.exists("best_agent.zip"):
    shutil.copy("best_agent.zip", "models/best_agent.zip")
    print("OK: best_agent.zip copiado a models/best_agent.zip")
elif os.path.exists("models/best_agent.zip"):
    print("OK: models/best_agent.zip ya existe")
else:
    raise FileNotFoundError(
        "No se encontró best_agent.zip. Debe estar en la raíz del proyecto "
        "o en models/best_agent.zip."
    )

!ls -lh models/best_agent.zip

In [ ]:
!env MPLBACKEND=Agg CUDA_VISIBLE_DEVICES="" xvfb-run -a {PY} eval.py --algo ppo --model models/best_agent --map Duckietown-loop_empty-v0 --episodes 1 --device cpu --init-order model-first

### Vídeo de evaluación del modelo final (opcional)

**Opcional y solo cualitativo.** Esta celda **no entrena nada**: carga `best_agent.zip` (= **PPO 20k**) y graba un MP4 del agente conduciendo en `Duckietown-loop_empty-v0`. Prueba varios *rollouts* y guarda el de **mayor recompensa** (para no mostrar un episodio sin movimiento). El vídeo se escribe en `outputs/best_agent_loop_empty.mp4` y se muestra embebido abajo. La **evaluación cuantitativa oficial** sigue siendo la celda anterior (`eval.py`).

In [ ]:
# OPCIONAL — Vídeo del modelo final conduciendo (NO entrena; carga best_agent.zip).
%cd $PROJECT

import base64
import os
import shutil
from IPython.display import HTML, display

os.makedirs("models", exist_ok=True)
if os.path.exists("best_agent.zip"):
    shutil.copy("best_agent.zip", "models/best_agent.zip")  # asegurar el modelo en models/

VIDEO_OUT = "outputs/best_agent_loop_empty.mp4"

# Un único rollout con seed 42: demostración reproducible del modelo final.
!env MPLBACKEND=Agg CUDA_VISIBLE_DEVICES="" xvfb-run -a {PY} scripts/make_eval_video.py --algo ppo --model models/best_agent --map Duckietown-loop_empty-v0 --out {VIDEO_OUT} --rollouts 1 --seed 42 --device cpu

# Mostrar el vídeo embebido (base64, self-contained).
if os.path.exists(VIDEO_OUT):
    _b64 = base64.b64encode(open(VIDEO_OUT, "rb").read()).decode()
    display(HTML(f'<video width="480" controls>'
                 f'<source src="data:video/mp4;base64,{_b64}" type="video/mp4">'
                 f'</video>'))
    print("OK: vídeo en", VIDEO_OUT)
else:
    print("No se generó el vídeo; revisa la salida de make_eval_video.py más arriba.")

---

## 📦 Requisitos de Entrega y Proceso de Evaluación

Estimados alumnos,

Para asegurar que el proceso de corrección sea justo y ágil, debéis seguir estrictamente las siguientes directrices para vuestra entrega final.

### 1. Entregables Obligatorios

Vuestra entrega debe ser un único archivo comprimido que contenga exactamente lo siguiente:

* **El Código Fuente:** Vuestro cuaderno `.ipynb` completo (o, en su defecto, los scripts de Python `train.py` y `eval.ipynb`).
* **Dependencias:** Un archivo `requirements.txt` con las versiones de las librerías fijadas **estrictamente** usando `==` (por ejemplo, `stable-baselines3==2.2.1`). Esto es vital para evitar problemas de compatibilidad.
* **El Agente Entrenado:** El archivo comprimido de vuestro modelo final (debe llamarse `best_duckie_agent.zip`).
* **La Memoria Técnica:** Un documento `Report.pdf`. Aquí debéis justificar teórica y empíricamente la elección y configuración de vuestro tercer algoritmo (Fase 3) y comparar detalladamente su rendimiento frente a las baselines obligatorias de DQN y PPO.

### 2. Cómo seréis evaluados (El Contrato de Evaluación)

Quiero ser muy transparente sobre cómo probaré vuestros proyectos: **no voy a depurar vuestro código ni a re-entrenar vuestros modelos desde cero.**

El proceso de corrección será exactamente el siguiente:

1. Tomaré vuestro archivo `best_duckie_agent.zip`.
2. Lo cargaré en mi propio script de evaluación, ejecutándolo en un entorno limpio basado estrictamente en vuestro `requirements.txt`.
3. Ejecutaré la celda de evaluación, pero cambiando el mapa al escenario oculto que no habéis visto durante el entrenamiento: **`Duckietown-loop_obstacles-v0`**.

Si habéis entrenado el modelo respetando las reglas de forma (mismas dimensiones de entrada, variables definidas y clases guardadas correctamente en el código), vuestro agente se cargará a la primera. Veré la simulación de vuestro coche intentando sortear los obstáculos y seréis puntuados por su capacidad de generalización y supervivencia en este nuevo mapa.

**⚠️ Advertencia final:** Si al intentar cargar vuestro modelo el script falla por incompatibilidad de versiones, clases no encontradas o errores de dimensiones en los tensores (*shape errors*), **la calificación de la parte práctica será un 0**.

Os recomiendo encarecidamente que abráis un Google Colab completamente en blanco, instaléis solo lo que dice vuestro `requirements.txt`, subáis vuestro `.zip` e intentéis ejecutar la evaluación vosotros mismos antes de enviar el trabajo definitivo.

## ¡Mucho ánimo con el entrenamiento y buena suerte!

---

### ✅ Cómo cumple esta entrega el contrato
El ZIP incluye: `Challenge_RL.ipynb`, **`requirements.txt`** (versiones con `==`), `q_learning_sandbox.py`, `train.py`, `eval.py`, `src/`, `scripts/`, **`best_agent.zip`** (en la raíz) y **`Report.pdf`**. Observación `(1,64,64)`→FrameStack(4)→`(4,64,64)`; `CustomCNN` y wrappers definidos en `src/` (importables al cargar). Mapa oculto `Duckietown-loop_obstacles-v0` solo se usa para evaluación, nunca para entrenar.

> **Nota de nombre:** el enunciado original menciona `best_duckie_agent.zip`; la presentación del reto pide **`best_agent.zip`**. Entregamos **`best_agent.zip`** (copia de `ppo_loop_empty_20k_gpu.zip`); `best_duckie_agent.zip` fue solo un nombre de desarrollo.